# Analytics Overview

**Le notebook présent sert à observer différentes métriques et visuels sur nos données**

    > à exécuter uniquement depuis Databricks ou Google Collab

In [0]:
# Dans une cellule, vérifie que le fichier est accessible
dbutils.fs.ls("/Volumes/main/default/raw/")

In [0]:
import os
import sys
os.getcwd()
import os
os.chdir("/Workspace/Users/karl.sondeji@aivancity.education/Spark-pipeline-on-Online-Retail")
os.getcwd()

**Exécution du pipeline**

In [0]:
from src.main import run_pipeline
run_pipeline()

**Importation des fonctions**

In [0]:
import matplotlib.pyplot as plt

from src.analytics.temporal_analysis import get_monthly_revenue
from src.analytics.customer_analysis import get_rfm_segment_summary, get_cohort_retention
from src.analytics.returns_analysis import get_return_rate_by_category
from src.analytics.pareto_analysis import get_customer_pareto, get_top_products


In [0]:
gold = "/Volumes/main/default/raw/gold"

In [0]:
bronze = "/Volumes/main/default/raw/bronze"

## Analyse temporelle

In [0]:
# --- Tendance mensuelle ---
monthly_pd = get_monthly_revenue(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_pd["year_month"], monthly_pd["total_revenue"], marker="o")
ax.set_title("Évolution du CA mensuel en millions")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [0]:
# --- Segments RFM ---
rfm_pd = get_rfm_segment_summary(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rfm_pd["rfm_segment"], rfm_pd["total_revenue"])
ax.set_title("CA par segment RFM")
plt.tight_layout()
plt.show()

In [0]:
# --- Pareto clients ---
pareto_pd = get_customer_pareto(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pareto_pd["rank"], pareto_pd["cumulative_pct"])
ax.axhline(80, color="red", linestyle="--", label="80% du CA")
ax.set_title("Courbe de Pareto — clients")
ax.set_xlabel("Nombre de clients (triés par CA décroissant)")
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT
  COUNT(*) AS nb_lignes,
  COUNT(DISTINCT CustomerID) AS nb_clients,
  COUNT(DISTINCT InvoiceNo) AS nb_factures
FROM delta.`/Volumes/main/default/raw/gold`;

Sur le graphique ci-dessus, on peut voir que les 1200 meilleurs clients sont responsable de 80% du CA sur près de 4312 clients.

In [0]:
# --- Taux de retour par catégorie ---
returns_pd = get_return_rate_by_category(spark, gold).toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(returns_pd["product_category"], returns_pd["return_rate_pct"])
ax.set_title("Taux de retour par catégorie")
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lower, count

In [0]:
# --- Top produits (tables) ---
top_revenue, top_volume = get_top_products(spark, gold, n=10)
print("Top 10 produits par CA:")
display(top_revenue)
print("Top 10 produits par volume:")
display(top_volume)